# FDI Structural Ties Development

This notebook develops the foreign direct investment component of the US-China Closeness Index.

It retains the completed IMF Direct Investment Positions investigation and adds the national outward-FDI workbook containing China Ministry of Commerce and US Bureau of Economic Analysis stocks by destination country.

The current production candidate is **anchor outward-investment presence in the comparison country**. IMF DIP remains available for validation, robustness and a possible later reverse-direction extension.

The raw IMF source is approximately 2.24 GB. Normal notebook runs therefore use cached repository-local intermediates rather than rescanning that source.

## Analytical objects

For each country, anchor and year, the source can describe two economic directions:

```text
anchor investment in country
country investment in anchor
```

Each direction may also be observed from two reporting perspectives:

```text
anchor report
country report
```

The source further distinguishes reported official data from values derived using counterparty information. These are alternative observations of bilateral positions, not independent ties to be summed.

## Primary candidate series

The first candidate set is restricted to:

```text
Instrument:        All financial instruments
Entity:            All entities
Outward entry:     Net assets less liabilities
Inward entry:      Net liabilities less assets
Unit:              US dollars, millions
```

Equity and debt series remain available for decomposition and robustness. They are not added to the all-instrument total.

## Current methodology status

The IMF investigation established that the all-instrument, all-entity bilateral series are net positions and that counterparty-derived observations mirror official reports. IMF DIP remains valuable for validation, but internal netting makes it an imperfect primary measure of investment depth.

The national workbook provides the preferred production candidate:

```text
China outward FDI stock in country
US outward FDI stock in country
```

This notebook therefore preserves the IMF diagnostics and adds a reproducible inventory of the national stock and GDP blocks before a final FDI score is fixed.

In [44]:
# locate repository and define paths
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd


def locate_repository_root(start=None):
    try:
        start_path = Path(start or Path.cwd()).resolve()
    except FileNotFoundError:
        start_path = Path.home()

    candidates = [start_path]
    candidates.extend(start_path.parents)

    for candidate in candidates:
        if (
            (candidate / "src" / "data").exists()
            and (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

    raise FileNotFoundError("Could not locate the repository root.")


REPOSITORY_ROOT = locate_repository_root()
RAW_INVENTORY_SCRIPT = REPOSITORY_ROOT / "src" / "data" / "inventory_imf_fdi.py"
CANDIDATE_SCRIPT = REPOSITORY_ROOT / "src" / "data" / "prepare_fdi_candidate_observations.py"
ANCHOR_SUBSET = REPOSITORY_ROOT / "data" / "interim" / "fdi" / "imf_dip_anchor_series.parquet"
CANDIDATES = REPOSITORY_ROOT / "data" / "interim" / "fdi" / "imf_dip_primary_candidates.parquet"
REPORTS = REPOSITORY_ROOT / "outputs" / "tables" / "fdi"

RUN_RAW_INVENTORY = False

print("Repository root located.")
print("Raw inventory script exists:", RAW_INVENTORY_SCRIPT.exists())
print("Candidate preparation script exists:", CANDIDATE_SCRIPT.exists())
print("Cached anchor subset exists:", ANCHOR_SUBSET.exists())

Repository root located.
Raw inventory script exists: True
Candidate preparation script exists: True
Cached anchor subset exists: True


In [45]:
# run the expensive raw inventory only when explicitly requested or missing
if RUN_RAW_INVENTORY or not ANCHOR_SUBSET.exists():
    completed = subprocess.run(
        [sys.executable, str(RAW_INVENTORY_SCRIPT)],
        cwd=REPOSITORY_ROOT,
        check=False,
        text=True,
        capture_output=True,
    )

    if completed.stdout.strip():
        print(completed.stdout)

    if completed.stderr.strip():
        print("INVENTORY STDERR:")
        print(completed.stderr)

    if completed.returncode != 0:
        raise RuntimeError(
            f"{RAW_INVENTORY_SCRIPT.name} failed with exit code "
            f"{completed.returncode}."
        )
else:
    print("Using cached IMF DIP anchor subset.")

Using cached IMF DIP anchor subset.


In [46]:
# prepare the primary long-form candidate observations
completed = subprocess.run(
    [sys.executable, str(CANDIDATE_SCRIPT)],
    cwd=REPOSITORY_ROOT,
    check=False,
    text=True,
    capture_output=True,
)

if completed.stdout.strip():
    print(completed.stdout)

if completed.stderr.strip():
    print("CANDIDATE SCRIPT STDERR:")
    print(completed.stderr)

if completed.returncode != 0:
    raise RuntimeError(
        f"{CANDIDATE_SCRIPT.name} failed with exit code "
        f"{completed.returncode}."
    )

IMF DIP PRIMARY FDI CANDIDATES
{
  "source_series_rows": 21550,
  "selected_primary_series_rows": 3012,
  "candidate_long_rows": 48192,
  "observed_candidate_values": 36104,
  "negative_candidate_values": 1838,
  "zero_candidate_values": 5686,
  "duplicate_candidate_keys": 0,
  "years": [
    2009,
    2024
  ],
  "economic_direction_counts": {
    "country_to_anchor": 19077,
    "anchor_to_country": 17027
  },
  "reporting_perspective_counts": {
    "anchor_report": 18107,
    "country_report": 17997
  },
  "data_type_counts": {
    "Reported official data": 18052,
    "Derived using counterparty information": 18052
  }
}

Candidate output: data/interim/fdi/imf_dip_primary_candidates.parquet
Reports directory: outputs/tables/fdi



In [47]:
# load candidate observations and reports
candidates = pd.read_parquet(CANDIDATES)
coverage = pd.read_csv(
    REPORTS / "imf_dip_primary_candidate_coverage.csv"
)
duplicates = pd.read_csv(
    REPORTS / "imf_dip_primary_candidate_duplicates.csv"
)

print("Candidate rows:", len(candidates))
print("Observed values:", int(candidates["observed"].sum()))
print("Years:", int(candidates["year"].min()), "to", int(candidates["year"].max()))
print("Duplicate candidate keys:", len(duplicates))

Candidate rows: 48192
Observed values: 36104
Years: 2009 to 2024
Duplicate candidate keys: 0


In [48]:
# validate primary dimensions and economic-direction assignment
assert candidates["INSTR_ASSET"].eq("All financial instruments").all()
assert candidates["DI_ENTITY"].eq("All entities").all()
assert candidates["year"].between(2009, 2024).all()
assert candidates["economic_direction"].isin(
    ["anchor_to_country", "country_to_anchor"]
).all()
assert candidates["reporting_perspective"].isin(
    ["anchor_report", "country_report"]
).all()

expected_accounting = {
    "Outward": "Net (assets less liabilities)",
    "Inward": "Net (liabilities less assets)",
}
assert candidates.apply(
    lambda row: row["ACCOUNTING_ENTRY"]
    == expected_accounting[row["DI_DIRECTION"]],
    axis=1,
).all()

print("PASS: all-instrument, all-entity net positions selected")
print("PASS: economic directions and reporting perspectives assigned")

PASS: all-instrument, all-entity net positions selected
PASS: economic directions and reporting perspectives assigned


In [49]:
# inspect candidate coverage
observed = candidates[candidates["observed"]].copy()

coverage_summary = (
    observed.groupby(
        [
            "anchor",
            "economic_direction",
            "reporting_perspective",
            "DV_TYPE",
        ]
    )
    .agg(
        observed_values=("fdi_position_usd_millions", "size"),
        countries=("country_source_name", "nunique"),
        negative_values=("negative", "sum"),
        zero_values=("zero", "sum"),
    )
    .reset_index()
)

display(coverage_summary)
display(coverage.tail(20))

,anchor,economic_direction,reporting_perspective,DV_TYPE,observed_values,countries,negative_values,zero_values
0,China,anchor_to_country,anchor_report,Derived using counterparty information,1708,130,79,68
1,China,anchor_to_country,anchor_report,Reported official data,1802,258,0,388
2,China,anchor_to_country,country_report,Derived using counterparty information,1795,257,0,388
3,China,anchor_to_country,country_report,Reported official data,1692,129,79,68
4,China,country_to_anchor,anchor_report,Derived using counterparty information,1191,99,21,144
5,China,country_to_anchor,anchor_report,Reported official data,3996,258,188,548
6,China,country_to_anchor,country_report,Derived using counterparty information,3980,257,188,548
7,China,country_to_anchor,country_report,Reported official data,1175,98,21,144
8,United States,anchor_to_country,anchor_report,Derived using counterparty information,1925,138,34,16
9,United States,anchor_to_country,anchor_report,Reported official data,3106,232,129,501


,year,anchor,economic_direction,reporting_perspective,DV_TYPE,series_rows,observed_values,countries,negative_values,zero_values
236,2023,United States,country_to_anchor,anchor_report,Derived using counterparty information,104,90,104,4,7
237,2023,United States,country_to_anchor,anchor_report,Reported official data,258,194,258,22,67
238,2023,United States,country_to_anchor,country_report,Derived using counterparty information,257,193,257,22,67
239,2023,United States,country_to_anchor,country_report,Reported official data,103,89,103,4,7
240,2024,China,anchor_to_country,anchor_report,Derived using counterparty information,134,106,134,4,2
241,2024,China,anchor_to_country,anchor_report,Reported official data,258,257,258,0,57
242,2024,China,anchor_to_country,country_report,Derived using counterparty information,257,256,257,0,57
243,2024,China,anchor_to_country,country_report,Reported official data,133,105,133,4,2
244,2024,China,country_to_anchor,anchor_report,Derived using counterparty information,100,76,100,1,7
245,2024,China,country_to_anchor,anchor_report,Reported official data,258,258,258,18,44


## Reporting hierarchy question

For a single economic direction, the dataset may contain:

- an official position reported by the investor side;
- an official position reported by the host side;
- a counterparty-derived value from either side.

The production hierarchy should select one value, not average or sum all available perspectives. The following cells compare coverage and disagreement before that hierarchy is fixed.

In [50]:
# compare official and counterparty-derived coverage
availability = (
    observed.assign(available=1)
    .pivot_table(
        index=[
            "country_source_name",
            "anchor",
            "year",
            "economic_direction",
            "reporting_perspective",
        ],
        columns="DV_TYPE",
        values="available",
        aggfunc="max",
        fill_value=0,
    )
    .reset_index()
)

for column in [
    "Reported official data",
    "Derived using counterparty information",
]:
    if column not in availability.columns:
        availability[column] = 0

availability["availability_profile"] = np.select(
    [
        availability["Reported official data"].eq(1)
        & availability["Derived using counterparty information"].eq(1),
        availability["Reported official data"].eq(1),
        availability["Derived using counterparty information"].eq(1),
    ],
    ["both", "official_only", "derived_only"],
    default="neither",
)

display(
    availability["availability_profile"]
    .value_counts()
    .rename_axis("profile")
    .reset_index(name="country_direction_reports")
)

,profile,country_direction_reports
0,both,9380
1,official_only,8672
2,derived_only,8672


In [51]:
# compare mirror reporting perspectives where both official values exist
official = observed[
    observed["DV_TYPE"].eq("Reported official data")
].copy()

mirror = official.pivot_table(
    index=[
        "country_source_name",
        "anchor",
        "year",
        "economic_direction",
    ],
    columns="reporting_perspective",
    values="fdi_position_usd_millions",
    aggfunc="first",
).reset_index()

mirror_pairs = mirror.dropna(
    subset=["anchor_report", "country_report"]
).copy()
mirror_pairs["absolute_difference"] = (
    mirror_pairs["anchor_report"]
    - mirror_pairs["country_report"]
).abs()
mirror_pairs["symmetric_denominator"] = (
    mirror_pairs["anchor_report"].abs()
    + mirror_pairs["country_report"].abs()
)
mirror_pairs["symmetric_relative_difference"] = np.where(
    mirror_pairs["symmetric_denominator"].gt(0),
    2 * mirror_pairs["absolute_difference"]
    / mirror_pairs["symmetric_denominator"],
    0,
)

print("Official mirror pairs:", len(mirror_pairs))
display(
    mirror_pairs[
        [
            "absolute_difference",
            "symmetric_relative_difference",
        ]
    ].describe()
)
display(
    mirror_pairs.nlargest(
        25,
        "absolute_difference",
    )
)

Official mirror pairs: 4667


reporting_perspective,absolute_difference,symmetric_relative_difference
count,4.667000e+03,4667.000000
mean,3.545739e+04,0.928846
std,1.591255e+05,0.699133
min,0.000000e+00,0.000000
25%,6.776906e+01,0.294058
50%,8.390249e+02,0.761593
75%,6.808984e+03,1.597423
max,2.002464e+06,2.000000


reporting_perspective,country_source_name,anchor,year,economic_direction,anchor_report,country_report,absolute_difference,symmetric_denominator,symmetric_relative_difference
13201,World,China,2024,anchor_to_country,3.139931e+06,1.137467e+06,2.002464e+06,4.277398e+06,0.936300
13200,World,China,2023,country_to_anchor,3.649817e+06,1.702045e+06,1.947772e+06,5.351862e+06,0.727886
13199,World,China,2023,anchor_to_country,2.955400e+06,1.034149e+06,1.921251e+06,3.989549e+06,0.963142
13202,World,China,2024,country_to_anchor,3.729411e+06,1.813711e+06,1.915700e+06,5.543122e+06,0.691199
13198,World,China,2022,country_to_anchor,3.569146e+06,1.677227e+06,1.891919e+06,5.246373e+06,0.721229
13196,World,China,2021,country_to_anchor,3.577743e+06,1.691743e+06,1.886001e+06,5.269486e+06,0.715820
13195,World,China,2021,anchor_to_country,2.785150e+06,9.205322e+05,1.864618e+06,3.705682e+06,1.006356
13197,World,China,2022,anchor_to_country,2.754814e+06,9.698016e+05,1.785013e+06,3.724616e+06,0.958495
13193,World,China,2020,anchor_to_country,2.580658e+06,8.693004e+05,1.711358e+06,3.449959e+06,0.992103
13194,World,China,2020,country_to_anchor,3.214115e+06,1.614413e+06,1.599701e+06,4.828528e+06,0.662604


In [52]:
# inspect negative and zero positions
negative = observed[observed["negative"]].copy()
zero = observed[observed["zero"]].copy()

print("Negative candidate observations:", len(negative))
print("Zero candidate observations:", len(zero))

display(
    negative.groupby(
        [
            "anchor",
            "economic_direction",
            "reporting_perspective",
            "DV_TYPE",
        ]
    )
    .size()
    .rename("negative_values")
    .reset_index()
)

display(
    negative.nsmallest(
        30,
        "fdi_position_usd_millions",
    )[
        [
            "country_source_name",
            "anchor",
            "year",
            "economic_direction",
            "reporting_perspective",
            "DV_TYPE",
            "fdi_position_usd_millions",
        ]
    ]
)

Negative candidate observations: 1838
Zero candidate observations: 5686


,anchor,economic_direction,reporting_perspective,DV_TYPE,negative_values
0,China,anchor_to_country,anchor_report,Derived using counterparty information,79
1,China,anchor_to_country,country_report,Reported official data,79
2,China,country_to_anchor,anchor_report,Derived using counterparty information,21
3,China,country_to_anchor,anchor_report,Reported official data,188
4,China,country_to_anchor,country_report,Derived using counterparty information,188
5,China,country_to_anchor,country_report,Reported official data,21
6,United States,anchor_to_country,anchor_report,Derived using counterparty information,34
7,United States,anchor_to_country,anchor_report,Reported official data,129
8,United States,anchor_to_country,country_report,Derived using counterparty information,129
9,United States,anchor_to_country,country_report,Reported official data,34


,country_source_name,anchor,year,economic_direction,reporting_perspective,DV_TYPE,fdi_position_usd_millions
7642,Belgium,United States,2011,anchor_to_country,anchor_report,Derived using counterparty information,-51433.689510
7645,Belgium,United States,2011,anchor_to_country,country_report,Reported official data,-51433.689510
10654,Belgium,United States,2012,anchor_to_country,anchor_report,Derived using counterparty information,-50116.617360
10657,Belgium,United States,2012,anchor_to_country,country_report,Reported official data,-50116.617360
22706,Belgium,United States,2016,country_to_anchor,anchor_report,Derived using counterparty information,-31892.217140
22709,Belgium,United States,2016,country_to_anchor,country_report,Reported official data,-31892.217140
4630,Belgium,United States,2010,anchor_to_country,anchor_report,Derived using counterparty information,-27953.304000
4633,Belgium,United States,2010,anchor_to_country,country_report,Reported official data,-27953.304000
25718,Belgium,United States,2017,country_to_anchor,anchor_report,Derived using counterparty information,-25495.558910
25721,Belgium,United States,2017,country_to_anchor,country_report,Reported official data,-25495.558910


## Negative-position policy

Negative direct-investment positions are valid net-accounting outcomes and must not be deleted as errors.

Before score construction, we will compare at least:

- retaining signed positions for audit while flooring structural magnitude at zero;
- a transform based on non-negative position magnitude;
- winsorization or robust scaling for extreme financial-centre observations; and
- sensitivity to excluding negative net positions from positive structural-depth credit.

No rule is fixed in this notebook yet.

In [53]:
# list source economies requiring mapping to the project universe
source_names = (
    candidates[["country_source_name"]]
    .drop_duplicates()
    .sort_values("country_source_name")
)

print("Distinct source economy names:", len(source_names))
display(source_names.head(100))

Distinct source economy names: 259


,country_source_name
0,"Afghanistan, Islamic Republic of"
4,Albania
12,Algeria
20,American Samoa
24,"Andorra, Principality of"
...,...
548,Guernsey
552,Guinea
558,Guinea-Bissau
566,Guyana


In [54]:
# inspect the 2024 candidate distributions
latest_year = int(observed["year"].max())
latest = observed[observed["year"].eq(latest_year)]

latest_summary = (
    latest.groupby(
        [
            "anchor",
            "economic_direction",
            "reporting_perspective",
            "DV_TYPE",
        ]
    )["fdi_position_usd_millions"]
    .describe()
    .reset_index()
)

display(latest_summary)

,anchor,economic_direction,reporting_perspective,DV_TYPE,count,mean,std,min,25%,50%,75%,max
0,China,anchor_to_country,anchor_report,Derived using counterparty information,106.0,32192.452827,150296.802254,-2022.357889,79.037048,969.517975,5147.979809,1.137467e+06
1,China,anchor_to_country,anchor_report,Reported official data,257.0,36652.891791,262273.851891,0.000000,1.533700,171.621400,1912.186300,3.139931e+06
2,China,anchor_to_country,country_report,Derived using counterparty information,256.0,36444.664141,262766.325234,0.000000,1.529025,167.875700,1869.901825,3.139931e+06
3,China,anchor_to_country,country_report,Reported official data,105.0,32175.438092,151017.551784,-2022.357889,73.912236,958.126448,4639.461807,1.137467e+06
4,China,country_to_anchor,anchor_report,Derived using counterparty information,76.0,71593.858426,287040.959876,-0.042357,16.111498,472.210640,9336.926343,1.813711e+06
5,China,country_to_anchor,anchor_report,Reported official data,258.0,43365.158072,313416.670733,-111.759846,0.021498,7.613496,409.684807,3.729411e+06
6,China,country_to_anchor,country_report,Derived using counterparty information,257.0,43304.406383,314026.693977,-111.759846,0.016480,7.534961,383.332654,3.729411e+06
7,China,country_to_anchor,country_report,Reported official data,75.0,70909.936538,288911.570646,-0.042357,15.302522,449.424612,7916.039752,1.813711e+06
8,United States,anchor_to_country,anchor_report,Derived using counterparty information,116.0,148706.853121,631519.596979,0.000000,423.858909,3510.862881,40158.206667,5.749998e+06
9,United States,anchor_to_country,anchor_report,Reported official data,183.0,74797.098361,519311.485522,-197.000000,5.500000,325.000000,9308.000000,6.826786e+06


# Part II: National Outward-FDI Sources

The workbook `data/sources/fdi/Outward Direct Investment.xlsx` contains China outward-investment stock, US outward-investment stock, a prototype US-minus-China calculation and World Bank nominal GDP.

The notebook conditionally runs `src/data/inventory_outward_fdi_workbook.py`. The prototype difference remains audit material only. Production development starts from the two separate anchor stocks and GDP fields.

In [55]:
# define national-source paths and execution policy
NATIONAL_SOURCE_WORKBOOK = (
    REPOSITORY_ROOT / "data" / "sources" / "fdi"
    / "Outward Direct Investment.xlsx"
)
NATIONAL_INVENTORY_SCRIPT = (
    REPOSITORY_ROOT / "src" / "data"
    / "inventory_outward_fdi_workbook.py"
)
NATIONAL_REPORTS = (
    REPOSITORY_ROOT / "outputs" / "tables"
    / "fdi_national_sources"
)
NATIONAL_INVENTORY_OUTPUTS = [
    NATIONAL_REPORTS / "outward_fdi_inventory_summary.json",
    NATIONAL_REPORTS / "outward_fdi_column_inventory.csv",
    NATIONAL_REPORTS / "outward_fdi_stock_long.csv",
    NATIONAL_REPORTS / "outward_fdi_coverage_by_year.csv",
    NATIONAL_REPORTS / "outward_fdi_duplicate_country_columns.csv",
]

# Set True only when a deliberate rebuild is required regardless of timestamps.
FORCE_NATIONAL_SOURCE_INVENTORY = False

print("National source workbook exists:", NATIONAL_SOURCE_WORKBOOK.exists())
print("National inventory script exists:", NATIONAL_INVENTORY_SCRIPT.exists())
print(
    "Cached national inventory complete:",
    all(path.exists() for path in NATIONAL_INVENTORY_OUTPUTS),
)

National source workbook exists: True
National inventory script exists: True
Cached national inventory complete: True


In [56]:
# refresh the inventory when forced, missing or older than its inputs
inventory_missing = any(
    not path.exists()
    for path in NATIONAL_INVENTORY_OUTPUTS
)

input_paths = [
    NATIONAL_SOURCE_WORKBOOK,
    NATIONAL_INVENTORY_SCRIPT,
]
inputs_exist = all(path.exists() for path in input_paths)

if not inputs_exist:
    missing_inputs = [
        str(path.relative_to(REPOSITORY_ROOT))
        for path in input_paths
        if not path.exists()
    ]
    raise FileNotFoundError(
        "Missing national-inventory input files: "
        + ", ".join(missing_inputs)
    )

outputs_stale = False
if not inventory_missing:
    newest_input_mtime = max(
        path.stat().st_mtime
        for path in input_paths
    )
    oldest_output_mtime = min(
        path.stat().st_mtime
        for path in NATIONAL_INVENTORY_OUTPUTS
    )
    outputs_stale = newest_input_mtime > oldest_output_mtime

run_national_inventory = (
    FORCE_NATIONAL_SOURCE_INVENTORY
    or inventory_missing
    or outputs_stale
)

if run_national_inventory:
    reasons = []
    if FORCE_NATIONAL_SOURCE_INVENTORY:
        reasons.append("forced rebuild")
    if inventory_missing:
        reasons.append("missing output")
    if outputs_stale:
        reasons.append("source or script is newer than cached outputs")

    print(
        "Refreshing national outward-FDI inventory:",
        "; ".join(reasons),
    )

    completed = subprocess.run(
        [sys.executable, str(NATIONAL_INVENTORY_SCRIPT)],
        cwd=REPOSITORY_ROOT,
        check=False,
        text=True,
        capture_output=True,
    )

    if completed.stdout.strip():
        print(completed.stdout)

    if completed.stderr.strip():
        print("NATIONAL INVENTORY STDERR:")
        print(completed.stderr)

    if completed.returncode != 0:
        raise RuntimeError(
            f"{NATIONAL_INVENTORY_SCRIPT.name} failed with exit code "
            f"{completed.returncode}."
        )
else:
    print("Using current national outward-FDI inventory outputs.")

Using current national outward-FDI inventory outputs.


In [57]:
# load national-source inventory outputs
import json

national_summary = json.loads(
    (NATIONAL_REPORTS / "outward_fdi_inventory_summary.json")
    .read_text(encoding="utf-8")
)
national_columns = pd.read_csv(
    NATIONAL_REPORTS / "outward_fdi_column_inventory.csv",
    low_memory=False,
)
national_long = pd.read_csv(
    NATIONAL_REPORTS / "outward_fdi_stock_long.csv",
    low_memory=False,
)
national_coverage = pd.read_csv(
    NATIONAL_REPORTS / "outward_fdi_coverage_by_year.csv",
    low_memory=False,
)
national_duplicates = pd.read_csv(
    NATIONAL_REPORTS / "outward_fdi_duplicate_country_columns.csv",
    low_memory=False,
)

print("Annual-sheet shape:", national_summary["annual_shape"])
print("Stock years:", national_summary["stock_years"])
print("Inventory columns:", len(national_columns))
print("Long stock rows:", len(national_long))
print("Duplicate country labels:", len(national_duplicates))

Annual-sheet shape: [203, 964]
Stock years: [1999, 2025]
Inventory columns: 963
Long stock rows: 26001
Duplicate country labels: 4


In [58]:
# validate workbook blocks and source units
expected_blocks = {
    "china_odi_stock",
    "us_odi_stock",
    "prototype_us_minus_china",
    "world_bank_nominal_gdp",
}
assert set(national_columns["block"].unique()) == expected_blocks
assert national_long["year"].between(1999, 2025).all()

source_units = (
    national_columns.groupby("block")["unit"]
    .agg(lambda values: sorted(set(values.dropna().astype(str))))
    .to_dict()
)
assert source_units["china_odi_stock"] == ["USDmn"]
assert source_units["us_odi_stock"] == ["USDmn"]
assert source_units["prototype_us_minus_china"] == ["USDmn"]
assert source_units["world_bank_nominal_gdp"] == ["USDbn"]

print("PASS: four expected workbook blocks found")
print("PASS: China and US stock blocks use USD millions")
print("PASS: GDP block uses USD billions")

PASS: four expected workbook blocks found
PASS: China and US stock blocks use USD millions
PASS: GDP block uses USD billions


In [59]:
# summarize observed, negative and zero values by block
block_quality = (
    national_long.groupby("block")
    .agg(
        rows=("year", "size"),
        observed_values=("observed", "sum"),
        negative_values=("negative", "sum"),
        zero_values=("zero", "sum"),
        source_names=("country_source_name", "nunique"),
    )
    .reset_index()
)
display(block_quality)
display(national_coverage.tail(20))

,block,rows,observed_values,negative_values,zero_values,source_names
0,china_odi_stock,5400,4084,1,12,192
1,prototype_us_minus_china,7101,4774,1790,609,262
2,us_odi_stock,6426,5529,291,303,233
3,world_bank_nominal_gdp,7074,6926,0,0,262


,block,year,columns,observed_values,negative_values,zero_values,economy_candidates
88,world_bank_nominal_gdp,2006,262,258,0,0,220
89,world_bank_nominal_gdp,2007,262,258,0,0,220
90,world_bank_nominal_gdp,2008,262,259,0,0,220
91,world_bank_nominal_gdp,2009,262,261,0,0,220
92,world_bank_nominal_gdp,2010,262,261,0,0,220
93,world_bank_nominal_gdp,2011,262,262,0,0,220
94,world_bank_nominal_gdp,2012,262,260,0,0,220
95,world_bank_nominal_gdp,2013,262,260,0,0,220
96,world_bank_nominal_gdp,2014,262,261,0,0,220
97,world_bank_nominal_gdp,2015,262,260,0,0,220


## Negative, zero and missing source states

The national source positions remain official directional investment positions, not gross corporate-asset inventories. The production panel will therefore retain signed stock, negative-position flags, reported-zero flags and source-observation flags separately.

A later structural-depth transformation may floor negative positions at zero, but it will not overwrite the signed source value.

In [60]:
# inspect duplicate country labels before harmonization
if national_duplicates.empty:
    print("No duplicated country labels within workbook blocks.")
else:
    display(national_duplicates)
    duplicate_details = national_columns.merge(
        national_duplicates[["block", "country_source_name"]],
        on=["block", "country_source_name"],
        how="inner",
        validate="many_to_one",
    ).sort_values(
        ["block", "country_source_name", "excel_column_index_zero_based"]
    )
    display(duplicate_details)

,block,country_source_name,columns
0,us_odi_stock,Eswatini,2
1,us_odi_stock,North Macedonia,2
2,us_odi_stock,South Korea,2
3,us_odi_stock,Timor-Leste,2


,block,source_label,excel_column_index_zero_based,country_source_name,unit,name_classification,first_observed_year,last_observed_year,observed_values,negative_values,zero_values
1,us_odi_stock,US Outward Direct Investment,267,Eswatini,USDmn,economy_candidate,1999.0,2025.0,27,1,0
6,us_odi_stock,US Outward Direct Investment,403,Eswatini,USDmn,economy_candidate,1999.0,2025.0,27,1,0
3,us_odi_stock,US Outward Direct Investment,323,North Macedonia,USDmn,economy_candidate,1999.0,2025.0,27,0,0
4,us_odi_stock,US Outward Direct Investment,353,North Macedonia,USDmn,economy_candidate,1999.0,2025.0,27,0,0
2,us_odi_stock,US Outward Direct Investment,308,South Korea,USDmn,economy_candidate,1999.0,2025.0,27,0,0
5,us_odi_stock,US Outward Direct Investment,393,South Korea,USDmn,economy_candidate,1999.0,2025.0,27,0,0
0,us_odi_stock,US Outward Direct Investment,260,Timor-Leste,USDmn,economy_candidate,2009.0,2025.0,17,0,12
7,us_odi_stock,US Outward Direct Investment,411,Timor-Leste,USDmn,economy_candidate,2009.0,2025.0,17,0,12


In [61]:
# inspect source coverage over the analytical period
analytical_coverage = national_coverage[
    national_coverage["year"].between(2000, 2024)
].copy()
coverage_pivot = analytical_coverage.pivot_table(
    index="year",
    columns="block",
    values="observed_values",
    aggfunc="first",
)
display(coverage_pivot)

for block in ["china_odi_stock", "us_odi_stock", "world_bank_nominal_gdp"]:
    observed_years = analytical_coverage.loc[
        analytical_coverage["block"].eq(block)
        & analytical_coverage["observed_values"].gt(0),
        "year",
    ]
    first_year = int(observed_years.min()) if not observed_years.empty else None
    last_year = int(observed_years.max()) if not observed_years.empty else None
    print(block, "observed range:", first_year, "to", last_year)

block,china_odi_stock,prototype_us_minus_china,us_odi_stock,world_bank_nominal_gdp
year,,,,
2000,0,0,183,252
2001,0,0,185,253
2002,0,0,185,257
2003,0,0,184,257
2004,155,217,196,257
2005,165,217,195,257
2006,174,217,194,258
2007,180,217,193,258
2008,181,217,195,259


china_odi_stock observed range: 2004 to 2024
us_odi_stock observed range: 2000 to 2024
world_bank_nominal_gdp observed range: 2000 to 2024


In [62]:
# inspect 2024 anchor stocks and GDP values
latest_national = national_long[
    national_long["year"].eq(2024)
    & national_long["observed"]
].copy()

for block in ["china_odi_stock", "us_odi_stock", "world_bank_nominal_gdp"]:
    block_latest = latest_national[
        latest_national["block"].eq(block)
        & latest_national["name_classification"].eq("economy_candidate")
    ].copy()
    print(block, "observations:", len(block_latest))
    display(
        block_latest.nlargest(20, "value")[[
            "country_source_name",
            "value",
            "unit",
            "negative",
            "zero",
        ]]
    )

china_odi_stock observations: 183


,country_source_name,value,unit,negative,zero
241,Hong Kong,1920809.36,USDmn,False,False
3913,Ukraine,567713.10,USDmn,False,False
4804,Trinidad & Tobago,331910.47,USDmn,False,False
4156,Bolivia,214933.81,USDmn,False,False
4831,Uruguay,116239.07,USDmn,False,False
970,Singapore,102641.42,USDmn,False,False
4939,Bermuda,38520.92,USDmn,False,False
3535,Montenegro,38422.28,USDmn,False,False
4966,Canada,34011.22,USDmn,False,False
3886,Switzerland,30575.22,USDmn,False,False


us_odi_stock observations: 203


,country_source_name,value,unit,negative,zero
5560,Andorra,6697705.0,USDmn,False,False
11392,United Kingdom,1004829.0,USDmn,False,False
9313,Netherlands,994682.0,USDmn,False,False
8665,Luxembourg,563017.0,USDmn,False,False
8071,Ireland,492652.0,USDmn,False,False
10393,Singapore,492646.0,USDmn,False,False
6451,Canada,450587.0,USDmn,False,False
6073,Bermuda,241650.0,USDmn,False,False
7558,Germany,216086.0,USDmn,False,False
10933,Switzerland,213159.0,USDmn,False,False


world_bank_nominal_gdp observations: 207


,country_source_name,value,unit,negative,zero
21652,IDA & IBRD Total,43660.23300,USDbn,False,False
21814,International Bank for Reconstruction & Develo...,40766.44300,USDbn,False,False
20842,EU,19501.07900,USDbn,False,False
20086,China,18729.66840,USDbn,False,False
21220,Germany,4685.59258,USDbn,False,False
22057,Japan,4190.00819,USDbn,False,False
21760,India,3760.81347,USDbn,False,False
22300,LAS (League of Arab States),3745.04377,USDbn,False,False
25513,United Kingdom,3695.53951,USDbn,False,False
21085,France,3160.44262,USDbn,False,False


In [63]:
# compare raw and log-transformed investment-stock distributions
stock = national_long[
    national_long["block"].isin(
        ["china_odi_stock", "us_odi_stock"]
    )
    & national_long["name_classification"].eq("economy_candidate")
    & national_long["year"].between(2000, 2024)
].copy()
stock["nonnegative_stock_usd_mn"] = stock["value"].clip(lower=0)
stock["log1p_nonnegative_stock"] = np.log1p(
    stock["nonnegative_stock_usd_mn"]
)

display(
    stock[stock["observed"]]
    .groupby("block")[[
        "value",
        "nonnegative_stock_usd_mn",
        "log1p_nonnegative_stock",
    ]]
    .describe()
)

value                                                       \
                  count          mean            std      min    25%     50%   
block                                                                          
china_odi_stock  3689.0   8670.987858   78949.894739     -0.2  16.91  164.69   
us_odi_stock     4675.0  44324.303102  342544.734475 -21251.0  10.00  296.00   

                                     nonnegative_stock_usd_mn                \
                     75%         max                    count          mean   
block                                                                         
china_odi_stock  1008.79  1920809.36                   3689.0   8670.987913   
us_odi_stock     6340.00  6697705.00                   4675.0  44340.143743   

                 ...                      log1p_nonnegative_stock            \
                 ...      75%         max                   count      mean   
block            ...                                                          
china_odi_stock  ...  1008.79  1920809.36                  3689.0  5.017298   
us_odi_stock     ...  6340.00  6697705.00                  4675.0  5.722427   

                                                                         
                      std  min       25%       50%       75%        max  
block                                                                    
china_odi_stock  2.750069  0.0  2.885359  5.110119  6.917498  14.468258  
us_odi_stock     3.757503  0.0  2.397895  5.693732  8.754792  15.717276  

[2 rows x 24 columns]

## Provisional production direction

The national workbook measures **anchor outward-investment presence in the comparison country**. It does not include the comparison country's investment in the anchor.

```text
National anchor sources
    preferred production candidate

IMF DIP
    validation, robustness and possible reverse-direction evidence
```

The next stage will resolve country mapping, source coverage, valuation differences and transformations before fixing the production score.

## Updated production sequence

1. Resolve duplicate labels and classify aggregates, territories and historical entities.
2. Map China stock, US stock and GDP names to the 199-country project universe.
3. Audit `n.s.`, blanks, zeros and negative values separately.
4. Decide how to handle years without a Chinese stock observation.
5. Compare national stocks with overlapping IMF anchor-to-country positions.
6. Test raw stock, `log1p` stock and stock-to-GDP intensity.
7. Decide whether absolute presence and host-economy intensity should be combined.
8. Build `fdi_structural_ties_country_anchor_year.csv`.
9. Retain IMF diagnostics and national-source provenance.

# Decision register

## Decided

- Keep Notebook 08 as the single FDI methodology notebook.
- Keep the large IMF scan in a chunked script and use cached intermediates normally.
- Retain the completed IMF net-position and mirror-report investigation.
- Treat IMF counterparty-derived observations as mirrors, not independent evidence.
- Use national China and US outward-FDI stocks as the preferred production candidate for anchor investment presence.
- Run the national workbook inventory conditionally from Notebook 08.
- Keep China and US stock measures separate until downstream orientation calculations.
- Preserve signed positions, zeros, negatives and missingness as distinct states.
- Exclude the prototype US-minus-China block from production inputs.
- Retain World Bank nominal GDP as a candidate intensity denominator.
- Keep IMF DIP for validation, robustness and possible reverse-direction evidence.

## Open

- Country-name harmonization and aggregate exclusions.
- Duplicate labels and historical geographies.
- Coverage policy for unavailable China years.
- Negative-position transformation.
- Raw-log stock versus stock-to-GDP intensity.
- Weighting absolute presence and host-economy intensity.
- Final scaling and later threshold selection, including Jenks analysis.